# Визуальная проверка bi-encoder retrieval

Для каждой из 19 размеченных пар (релевантный товар → пост) ноутбук рендерит HTML-блок:
- заголовок пары + ранг, на котором каждая модель нашла целевой пост (или «НЕ НАЙДЕНО»);
- описание товара (query);
- целевой пост;
- TOP-K кандидатов от RoSBERTa (полные тексты, совпавший с целью — зелёный);
- TOP-K кандидатов от E5-base (полные тексты, совпавший с целью — зелёный).

Тексты не обрезаются. Чистый bi-encoder, без cross-encoder.


## 1. Импорты и настройки

In [1]:
import warnings
warnings.filterwarnings('ignore')

import json
import html
import torch
import lancedb
from sentence_transformers import SentenceTransformer
from IPython.display import HTML, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")


Device: cpu


In [2]:
# ==================== НАСТРОЙКИ ====================
BI_ENCODER_ROSBERT_PATH = "models/final/bi-encoder"
BI_ENCODER_E5_PATH      = "intfloat/multilingual-e5-base"

LANCEDB_PATH  = "./lancedb_store"
TABLE_ROSBERT = "rosberta-fine-tuned-50k"    # дообученный RoSBERTa (384d)
TABLE_E5      = "e5-base-base-50k"   # E5-base zero-shot (768d)

GT_POSTS_JSON = "ground_truth_posts.json"
GT_PAIRS_JSON = "ground_truth_pairs.json"

TOP_K = 20  # сколько постов показывать на каждый запрос


## 2. Ground truth, модели, базы

In [3]:
with open(GT_POSTS_JSON, encoding='utf-8') as f:
    gt_posts = json.load(f)
with open(GT_PAIRS_JSON, encoding='utf-8') as f:
    gt_pairs = json.load(f)

post_text_by_num = {p['post_id']: p['text'] for p in gt_posts}

print(f'GT постов: {len(gt_posts)}')
print(f'GT пар (товар → пост): {len(gt_pairs)}')


GT постов: 10
GT пар (товар → пост): 19


In [4]:
print('Загружаем RoSBERTa (fine-tuned)...')
bi_rosbert = SentenceTransformer(BI_ENCODER_ROSBERT_PATH, device=device)
print(f'  dim={bi_rosbert.get_sentence_embedding_dimension()}')

print('Загружаем E5-base (zero-shot)...')
bi_e5 = SentenceTransformer(BI_ENCODER_E5_PATH, device=device)
print(f'  dim={bi_e5.get_sentence_embedding_dimension()}')

db = lancedb.connect(LANCEDB_PATH)
table_ros = db.open_table(TABLE_ROSBERT)
table_e5  = db.open_table(TABLE_E5)
print(f'\nТаблица {TABLE_ROSBERT}: {table_ros.count_rows():,} записей')
print(f'Таблица {TABLE_E5}: {table_e5.count_rows():,} записей')


Загружаем RoSBERTa (fine-tuned)...
  dim=384
Загружаем E5-base (zero-shot)...
  dim=768

Таблица posts: 50,010 записей
Таблица posts3: 50,010 записей


## 3. Поиск и HTML-рендер

In [5]:
def encode_rosbert(q):
    return bi_rosbert.encode([q], normalize_embeddings=True)[0].tolist()

def encode_e5(q):
    return bi_e5.encode(['query: ' + q], normalize_embeddings=True)[0].tolist()

def search(table, qvec, k):
    return (table.search(qvec, query_type='vector')
                 .limit(k)
                 .select(['text', 'channel', 'category'])
                 .to_list())

def esc(s):
    return html.escape(str(s)).replace('\n', '<br>')

def find_rank(results, target_text):
    t = target_text.strip()
    for i, r in enumerate(results, 1):
        if r['text'].strip() == t:
            return i
    return None

def rank_badge(rank, label, bg_color):
    if rank is None:
        return (f'<span style="background:#f8d7da;color:#000;padding:6px 12px;'
                f'border-radius:4px;font-weight:bold;border:1px solid #f1aeb5;">'
                f'{label}: не найдено в TOP-{TOP_K}</span>')
    return (f'<span style="background:{bg_color};color:#000;padding:6px 12px;'
            f'border-radius:4px;font-weight:bold;border:1px solid #adb5bd;">'
            f'{label}: ранг #{rank}</span>')

def card(rank, post, is_target, accent_bg):
    if is_target:
        bg = '#d4edda'; border = '#28a745'; badge_bg = '#c3e6cb'
        star = ' ★ ЦЕЛЬ'
    else:
        bg = '#f8f9fa'; border = '#dee2e6'; badge_bg = accent_bg
        star = ''
    return (
        f'<div style="background:{bg};border-left:4px solid {border};'
        f'padding:12px 14px;margin:8px 0;border-radius:4px;color:#000;">'
        f'<div style="margin-bottom:8px;font-size:13px;color:#000;">'
        f'<span style="background:{badge_bg};color:#000;padding:3px 9px;'
        f'border-radius:3px;font-weight:bold;border:1px solid #adb5bd;">#{rank}</span>'
        f'<span style="color:#000;margin-left:10px;">'
        f'<b>@{esc(post["channel"])}</b> · {esc(post.get("category",""))}'
        f'{star}</span></div>'
        f'<div style="white-space:pre-wrap;font-family:system-ui,sans-serif;'
        f'font-size:14px;color:#000;line-height:1.5;">{esc(post["text"])}</div>'
        f'</div>'
    )

def render_pair(idx, total, pair, target_text, res_r, res_e):
    rank_r = find_rank(res_r, target_text)
    rank_e = find_rank(res_e, target_text)

    cards_r = ''.join(
        card(i, r, r['text'].strip() == target_text.strip(), '#cfe2ff')
        for i, r in enumerate(res_r, 1)
    )
    cards_e = ''.join(
        card(i, r, r['text'].strip() == target_text.strip(), '#ffe5d0')
        for i, r in enumerate(res_e, 1)
    )

    return HTML(f'''
    <div style="border:2px solid #495057;border-radius:8px;margin:28px 0;
                background:#ffffff;font-family:system-ui,sans-serif;
                overflow:hidden;color:#000;">

        <div style="background:#e9ecef;color:#000;padding:14px 20px;
                    border-bottom:1px solid #ced4da;">
            <div style="font-size:18px;font-weight:bold;color:#000;">
                Пара {idx}/{total} · GT post #{pair["post_num"]}
            </div>
            <div style="font-size:13px;color:#000;margin-top:4px;">
                {esc(pair.get("imt_name",""))} · {esc(pair.get("subj_name",""))}
            </div>
        </div>

        <div style="padding:16px 20px;background:#f8f9fa;
                    display:flex;gap:14px;flex-wrap:wrap;
                    border-bottom:1px solid #e9ecef;">
            {rank_badge(rank_r, "RoSBERTa", "#cfe2ff")}
            {rank_badge(rank_e, "E5-base", "#ffe5d0")}
        </div>

        <div style="padding:20px;background:#ffffff;color:#000;">
            <div style="background:#e7f3ff;border-left:4px solid #0d6efd;
                        padding:12px 14px;margin-bottom:14px;border-radius:4px;color:#000;">
                <div style="font-weight:bold;color:#000;margin-bottom:6px;
                            font-size:13px;text-transform:uppercase;letter-spacing:0.5px;">
                    Запрос (описание товара)
                </div>
                <div style="white-space:pre-wrap;font-size:14px;line-height:1.5;color:#000;">
                    {esc(pair["description"])}
                </div>
            </div>

            <div style="background:#d4edda;border-left:4px solid #28a745;
                        padding:12px 14px;margin-bottom:20px;border-radius:4px;color:#000;">
                <div style="font-weight:bold;color:#000;margin-bottom:6px;
                            font-size:13px;text-transform:uppercase;letter-spacing:0.5px;">
                    Целевой пост (должен найтись)
                </div>
                <div style="white-space:pre-wrap;font-size:14px;line-height:1.5;color:#000;">
                    {esc(target_text)}
                </div>
            </div>

            <h4 style="color:#000;border-bottom:2px solid #0d6efd;
                       padding-bottom:6px;margin:24px 0 8px 0;">
                RoSBERTa — TOP {TOP_K}
            </h4>
            {cards_r}

            <h4 style="color:#000;border-bottom:2px solid #fd7e14;
                       padding-bottom:6px;margin:28px 0 8px 0;">
                E5-base — TOP {TOP_K}
            </h4>
            {cards_e}
        </div>
    </div>
    ''')


## 4. Сводка: где нашёлся целевой пост по каждой паре

In [6]:
# Сначала быстрая сводная таблица: рангы по всем парам
summary_rows = []
for idx, pair in enumerate(gt_pairs, 1):
    target_text = post_text_by_num[pair['post_num']]

    qvec_r = encode_rosbert(pair['description'])
    qvec_e = encode_e5(pair['description'])
    res_r = search(table_ros, qvec_r, TOP_K)
    res_e = search(table_e5,  qvec_e, TOP_K)

    summary_rows.append({
        'idx': idx,
        'pair': pair,
        'target_text': target_text,
        'res_r': res_r,
        'res_e': res_e,
        'rank_r': find_rank(res_r, target_text),
        'rank_e': find_rank(res_e, target_text),
    })

# HTML-сводка
def fmt_rank(r):
    if r is None:
        return ('<span style="background:#f8d7da;color:#000;padding:2px 8px;'
                'border-radius:3px;font-weight:bold;border:1px solid #f1aeb5;">—</span>')
    if r <= 5:
        bg = '#d4edda'; bd = '#28a745'
    elif r <= 20:
        bg = '#fff3cd'; bd = '#ffc107'
    else:
        bg = '#f8d7da'; bd = '#f1aeb5'
    return (f'<span style="background:{bg};color:#000;padding:2px 8px;'
            f'border-radius:3px;font-weight:bold;border:1px solid {bd};">#{r}</span>')

rows_html = ''.join(
    f'<tr style="background:#ffffff;color:#000;">'
    f'<td style="padding:6px 10px;color:#000;border-bottom:1px solid #e9ecef;">{row["idx"]}</td>'
    f'<td style="padding:6px 10px;color:#000;border-bottom:1px solid #e9ecef;">GT #{row["pair"]["post_num"]}</td>'
    f'<td style="padding:6px 10px;color:#000;border-bottom:1px solid #e9ecef;">{esc(row["pair"].get("imt_name",""))[:70]}</td>'
    f'<td style="padding:6px 10px;text-align:center;border-bottom:1px solid #e9ecef;">{fmt_rank(row["rank_r"])}</td>'
    f'<td style="padding:6px 10px;text-align:center;border-bottom:1px solid #e9ecef;">{fmt_rank(row["rank_e"])}</td></tr>'
    for row in summary_rows
)

hits_r = sum(1 for r in summary_rows if r['rank_r'] is not None)
hits_e = sum(1 for r in summary_rows if r['rank_e'] is not None)

display(HTML(f'''
<div style="font-family:system-ui,sans-serif;margin:16px 0;color:#000;">
    <div style="font-size:16px;margin-bottom:10px;color:#000;">
        <b>Hit rate в TOP-{TOP_K}:</b>
        RoSBERTa <span style="color:#000;font-weight:bold;background:#cfe2ff;padding:2px 8px;border-radius:3px;">{hits_r}/{len(summary_rows)}</span>,
        E5-base <span style="color:#000;font-weight:bold;background:#ffe5d0;padding:2px 8px;border-radius:3px;">{hits_e}/{len(summary_rows)}</span>
    </div>
    <table style="border-collapse:collapse;font-size:13px;width:100%;background:#ffffff;color:#000;">
        <thead>
            <tr style="background:#e9ecef;color:#000;">
                <th style="padding:8px 10px;text-align:left;color:#000;border-bottom:2px solid #adb5bd;">#</th>
                <th style="padding:8px 10px;text-align:left;color:#000;border-bottom:2px solid #adb5bd;">GT пост</th>
                <th style="padding:8px 10px;text-align:left;color:#000;border-bottom:2px solid #adb5bd;">Товар</th>
                <th style="padding:8px 10px;color:#000;border-bottom:2px solid #adb5bd;">RoSBERTa</th>
                <th style="padding:8px 10px;color:#000;border-bottom:2px solid #adb5bd;">E5-base</th>
            </tr>
        </thead>
        <tbody>
            {rows_html}
        </tbody>
    </table>
    <div style="font-size:11px;color:#000;margin-top:6px;">
        Зелёный — ранг ≤5, жёлтый — ≤20, красный — не найдено в TOP-{TOP_K}
    </div>
</div>
'''))


#,GT пост,Товар,RoSBERTa,E5-base
1,GT #1,Затирка для плитки готовая - белая,#3,#1
2,GT #1,Самоклеящиеся панели для стен на кухню 60х30см пвх 15шт,—,—
3,GT #2,Развивашки 2-3-4 года/пиши стирай тетрадь/книги для малышей,—,—
4,GT #2,"Детская мозаика (5 цветов, 40 элементов) ""Кораблик""",—,—
5,GT #3,Жиросжигатель для похудения женщинам 60 капсул,—,—
6,GT #3,Таблетки для похудения - Эффективный жиросжигатель,—,—
7,GT #4,Накидка на сиденье DongFeng Fengshen Yixuan GS,—,—
8,GT #4,Кроссовер Monjaro,—,#13
9,GT #5,Матрас надувной двуспальный 203х152см с подушками и насосом,—,—
10,GT #5,Гуд Найт Мягкое фито снотворное для сна,—,—


## 5. Подробный просмотр всех пар

Проскролль вниз, читай глазами. Совпавший с целью пост выделен зелёным.

In [7]:
for row in summary_rows:
    display(render_pair(
        row['idx'], len(summary_rows), row['pair'],
        row['target_text'], row['res_r'], row['res_e']
    ))
